In [ ]:
"""
This notebook aims to determine the manifold of text embeddings of CLIP, wether they correspond to the <eos> token,
or token placed before in the sentence

"""

In [ ]:
!pip install datasets
!pip install pycocotools

In [ ]:
"""
Computes mean norm and its variance for embeddings placed at index 5 in a sentence, among MS-COCO dataset

"""

In [ ]:
def analyze_5th_embedding(tokenizer, text_encoder, centered, device="cuda"):
    dataset = load_dataset("lmms-lab/COCO-Caption2017", split='val', streaming=True)

    hidden_dim = text_encoder.config.hidden_size
    mean_vec = torch.zeros(hidden_dim, device=device)

    # --- Étape 1 : Calcul du vecteur moyen ---
    embs = []
    for i, example in enumerate(tqdm(dataset, total=1000, desc="Collecting embeddings")):
        prompt = example["answer"][0]
        enc = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
        tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
        with torch.no_grad():
            out = text_encoder(**enc).last_hidden_state  # [batch, seq_len, hidden_dim]
            if tokens[5] == '<|endoftext|>' :
              emb = out[0, 1]  # embedding du token à l’indice 5
              print("fin de texte avant 5")
            else :
              emb = out[0, 5]
            embs.append(emb)
            mean_vec += emb / 1000

        if i >= 999:
            break

    # --- Étape 2 : Soustraction du vecteur moyen et calcul des normes ---
    norms = []
    for i, emb in enumerate(tqdm(embs, total=1000, desc="Computing norms")):
        if centered :
          centered = emb - mean_vec
        else :
          center = emb
        norm = center.norm(p=2).item()
        norms.append(norm)

    norms = torch.tensor(norms, device=device)
    mean_norm = norms.mean().item()
    var_norm = norms.var(unbiased=False).item()

    return mean_vec, mean_norm, var_norm


In [ ]:
mean_vec, mean_norm, var_norm = analyze_5th_embedding(pipe.tokenizer, pipe.text_encoder,centered = False)
print(mean_norm)
print(var_norm)

In [ ]:
"""
Computes mean norm and its variance for embeddings corresponding to the <eos> token in a sentence, among MS-COCO dataset

"""

In [ ]:
def analyze_final_embedding(tokenizer, text_encoder, centered, device="cuda"):
    dataset = load_dataset("sayakpaul/coco-30-val-2014", split='train', streaming=True)

    hidden_dim = text_encoder.config.hidden_size
    mean_vec = torch.zeros(hidden_dim, device=device)

    # --- Étape 1 : Calcul du vecteur moyen ---
    embs = []
    for i, example in enumerate(tqdm(dataset, total=1000, desc="Collecting embeddings")):
        prompt = example["caption"]
        enc = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
        tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
        for k,token in (enumerate(tokens)):
          if token == '<|endoftext|>' :
            indice = k
            break
        if i < 10 :
          print(indice)
        with torch.no_grad():
            out = text_encoder(**enc)  # [batch, seq_len, hidden_dim]
            emb = out.pooler_output.squeeze(0)
            embs.append(emb)
            mean_vec += emb / 1000

        if i >= 999:
            break

    # --- Étape 2 : Soustraction du vecteur moyen et calcul des normes ---
    norms = []
    for i, emb in enumerate(tqdm(embs, total=1000, desc="Computing norms")):
        if centered :
          center = emb - mean_vec
        else :
          center = emb
        norm = center.norm(p=2).item()
        norms.append(norm)

    norms = torch.tensor(norms, device=device)
    mean_norm = norms.mean().item()
    var_norm = norms.var(unbiased=False).item()

    return mean_vec, mean_norm, var_norm


In [ ]:
mean_vec, mean_norm, var_norm = analyze_final_embedding(pipe.tokenizer, pipe.text_encoder,centered = False)
print(mean_norm)
print(var_norm)